# Notebook 4: Fama-French Factor Model

In Notebooks 2 and 3 we showed that the value premium exists but has weakened.
Now we ask a deeper question: is the value premium "real" alpha, or is it just 
compensation for known risks?

We test this by regressing our BM and EP spreads on the Fama-French three factors:

- **Mkt-RF:** The overall market return above the risk-free rate
- **SMB (Small Minus Big):** Returns from owning small stocks vs large stocks
- **HML (High Minus Low):** This IS the value factor — it literally measures the 
  value premium

If our spread loads entirely on HML with zero alpha, it means our value premium 
is fully explained by known risk factors and is not a separate phenomenon.
If alpha is significant, we have found something extra beyond what the 
three-factor model captures.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

bm_wide = pd.read_csv('bm_wide.csv', index_col='date', parse_dates=True)
ep_wide = pd.read_csv('ep_wide.csv', index_col='date', parse_dates=True)

ff3 = pd.read_csv('ff3_factors.csv', index_col='Date', parse_dates=True)

print("FF3 columns:", ff3.columns.tolist())
print(ff3.head())

FF3 columns: ['Mkt-RF', 'SMB', 'HML', 'RF']
            Mkt-RF     SMB     HML      RF
Date                                      
1963-01-01  0.0494  0.0301  0.0224  0.0025
1963-02-01 -0.0240  0.0046  0.0215  0.0023
1963-03-01  0.0308 -0.0258  0.0210  0.0023
1963-04-01  0.0451 -0.0126  0.0100  0.0025
1963-05-01  0.0176  0.0108  0.0255  0.0024


### Fama-French Three-Factor Regression

We regress our BM and EP spreads on the three Fama-French factors.
The key things to look for:
- HML loading: should be large and positive (our spread IS a value strategy)
- Alpha (const): if significant, we have returns beyond what factors explain
- R-squared: how much of our spread is explained by the three factors

In [ ]:
# change ff3 dates to end of month to match the rest 
ff3.index = ff3.index + pd.offsets.MonthEnd(0)

print("FF3 dates after fix:", ff3.index[:3])

FF3 dates after fix: DatetimeIndex(['1963-01-31', '1963-02-28', '1963-03-31'], dtype='datetime64[ns]', name='Date', freq=None)


In [10]:
bm_spread = bm_wide[['spread']].copy()
ep_spread = ep_wide[['spread']].copy()

bm_spread['spread'] = bm_spread['spread'] / 100
ep_spread['spread'] = ep_spread['spread'] / 100

bm_merged = bm_spread.join(ff3[['Mkt-RF','SMB','HML']], how='inner')
ep_merged = ep_spread.join(ff3[['Mkt-RF','SMB','HML']], how='inner')

print("BM merged shape:", bm_merged.shape)
print("EP merged shape:", ep_merged.shape)
print(bm_merged.head())

BM merged shape: (519, 4)
EP merged shape: (519, 4)
              spread  Mkt-RF     SMB     HML
1963-01-31  0.039413  0.0494  0.0301  0.0224
1963-02-28  0.025087 -0.0240  0.0046  0.0215
1963-04-30  0.000725  0.0451 -0.0126  0.0100
1963-05-31  0.035323  0.0176  0.0108  0.0255
1963-07-31 -0.019479 -0.0039 -0.0057 -0.0081


In [ ]:
X_bm = sm.add_constant(bm_merged[['Mkt-RF','SMB','HML']])
model_bm_ff = sm.OLS(bm_merged['spread'], X_bm).fit()

X_ep = sm.add_constant(ep_merged[['Mkt-RF','SMB','HML']])
model_ep_ff = sm.OLS(ep_merged['spread'], X_ep).fit()

print("=== FAMA-FRENCH FACTOR REGRESSION RESULTS ===\n")

for name, model in [('BM', model_bm_ff), ('EP', model_ep_ff)]:
    print(f"--- {name} Spread ---")
    print(f"  Alpha (const): {model.params['const']:.4f}  "
          f"(t = {model.tvalues['const']:.3f}, p = {model.pvalues['const']:.3f})")
    print(f"  Mkt-RF:        {model.params['Mkt-RF']:.4f}  "
          f"(t = {model.tvalues['Mkt-RF']:.3f})")
    print(f"  SMB:           {model.params['SMB']:.4f}  "
          f"(t = {model.tvalues['SMB']:.3f})")
    print(f"  HML:           {model.params['HML']:.4f}  "
          f"(t = {model.tvalues['HML']:.3f})")
    print(f"  R-squared:     {model.rsquared:.4f}")
    print()

=== FAMA-FRENCH FACTOR REGRESSION RESULTS ===

--- BM Spread ---
  Alpha (const): 0.0040  (t = 3.684, p = 0.000)
  Mkt-RF:        -0.1033  (t = -4.136)
  SMB:           0.3842  (t = 10.873)
  HML:           0.6889  (t = 18.761)
  R-squared:     0.4648

--- EP Spread ---
  Alpha (const): 0.0005  (t = 0.507, p = 0.613)
  Mkt-RF:        0.0453  (t = 1.870)
  SMB:           0.1853  (t = 5.410)
  HML:           0.6762  (t = 18.993)
  R-squared:     0.4176



### Interpretation: Factor Model Results

Both BM and EP spreads load heavily on HML (0.69 and 0.68 respectively), 
confirming that our value strategies behave like the academic value factor.

The key difference is in alpha:
- BM generates statistically significant alpha of 0.40% per month (t = 3.684, 
  p = 0.000) even after controlling for market, size, and value risk factors. 
  This suggests BM captures something beyond what the three-factor model explains.
  
- EP generates no significant alpha (t = 0.507, p = 0.613). Its returns are 
  fully explained by the Fama-French factors.

The R-squared of ~46% for BM and ~42% for EP means the three-factor model 
explains less than half the variation in our value spreads — confirming that 
value investing remains only partially understood.